# 02. Construcción del dataset: cobertura y panel 2022-2026

Evalúa la cobertura de las 19 métricas descargadas (por variable, conjunta y por decil de popularidad) y construye el panel final: 5 variables, 49 meses (marzo 2022 a marzo 2026) y artistas del marco top 5000 con datos en al menos el 80% de los meses.

**Entrada:** mensual consolidado e insumos del notebook 01 (`chartmetric_pilotos/`).
**Salida:** `artistas_ST_<n panel>de<n marco>_top5000_22_26_na_long.csv` y `_wide.csv`, y la cobertura por artista `artistas_ST_<n descargados>_22_26.csv`.

In [ ]:
# PAQUETES

library(data.table)
library(ggplot2)

## 1. Configuración y carga

In [ ]:
# ============================================================
# CONFIGURACIÓN Y CARGA DE INSUMOS DEL NOTEBOOK 01
# ============================================================

OUT_DIR <- "chartmetric_pilotos"
MONTHLY_FILE <- file.path(OUT_DIR, "chartmetric_monthly_2021_2026.rds")
INSUMOS_NOTEBOOK_02_FILE <- file.path(OUT_DIR, "insumos_notebook_02.rds")
FIGURES_DIR <- "figuras"

dir.create(FIGURES_DIR, showWarnings = FALSE, recursive = TRUE)

stopifnot(
  "Falta el mensual consolidado: correr el notebook 01" = file.exists(MONTHLY_FILE),
  "Faltan los insumos: correr el notebook 01" = file.exists(INSUMOS_NOTEBOOK_02_FILE)
)

insumos_notebook_02 <- readRDS(INSUMOS_NOTEBOOK_02_FILE)

FAN_SINCE <- insumos_notebook_02$fan_since
END_DATE <- insumos_notebook_02$end_date
REQUIRED_VARIABLES <- insumos_notebook_02$required_variables
PILOT2_ARTIST_IDS <- insumos_notebook_02$muestra_ids
ARTISTS <- insumos_notebook_02$artistas
rank_data <- insumos_notebook_02$rank_deciles_7000
MARCO_TOP_N <- insumos_notebook_02$marco_top_n
MARCO_TOP <- insumos_notebook_02$marco_top

# Subconjuntos por familia de endpoint
FAN_REQUIRED_VARIABLES <- REQUIRED_VARIABLES[grepl("^fan_stats__", REQUIRED_VARIABLES)]
PLAYLIST_REQUIRED_VARIABLES <- REQUIRED_VARIABLES[grepl("^playlist_evolution__", REQUIRED_VARIABLES)]

# Línea de referencia en las figuras de cobertura (%)
COVERAGE_THRESHOLD <- 70

pilot2_monthly <- as.data.table(readRDS(MONTHLY_FILE))[
  artist_id %in% PILOT2_ARTIST_IDS
]

In [ ]:
# ------------------------------------------------------------
# SANITY CHECK DE LA CARGA
# ------------------------------------------------------------

cat(
  "Insumos generados:", format(insumos_notebook_02$fecha_generacion), "\n",
  "Artistas de la muestra:", length(PILOT2_ARTIST_IDS), "\n",
  "Artistas con datos mensuales:", uniqueN(pilot2_monthly$artist_id), "\n",
  "Período:", format(min(pilot2_monthly$month)), "a", format(max(pilot2_monthly$month)), "\n",
  "Filas:", nrow(pilot2_monthly), "\n",
  "Variables objetivo ausentes:", length(setdiff(REQUIRED_VARIABLES, unique(pilot2_monthly$variable))), "\n"
)

stopifnot(
  "Hay artistas de la muestra sin datos mensuales" =
    setequal(unique(pilot2_monthly$artist_id), PILOT2_ARTIST_IDS)
)

## 2. Cobertura mensual por variable

In [ ]:
# ============================================================
# COBERTURA MENSUAL INTEGRADA
# Fan Stats + Playlist Evolution
#
# coverage_all_artists_pct:
#   porcentaje sobre todos los artistas de la muestra.
#
# coverage_available_artists_pct:
#   porcentaje entre artistas que tuvieron esa variable
#   al menos una vez durante el período.
# ============================================================

build_integrated_monthly_coverage <- function(
  monthly,
  ids,
  variables,
  since,
  until
) {

  ids <- unique(
    as.character(ids)
  )

  variables <- unique(
    as.character(variables)
  )

  first_month <- as.Date(
    format(
      since,
      "%Y-%m-01"
    )
  )

  last_month <- as.Date(
    format(
      until,
      "%Y-%m-01"
    )
  )

  months <- seq(
    first_month,
    last_month,
    by = "month"
  )

  n_artists <- length(ids)

  d <- copy(
    monthly[
      artist_id %in% ids &
        variable %in% variables &
        month >= first_month &
        month <= last_month
    ]
  )

  d[
    ,
    artist_id := as.character(artist_id)
  ]

  d[
    ,
    month := as.Date(month)
  ]

  if (
    !"interpolated" %in% names(d)
  ) {

    d[
      ,
      interpolated := FALSE
    ]
  }

  if (
    !"extrapolated" %in% names(d)
  ) {

    d[
      ,
      extrapolated := FALSE
    ]
  }

  # Una sola fila por artista, mes y variable.
  d <- unique(
    d,
    by = c(
      "artist_id",
      "month",
      "variable"
    ),
    fromLast = TRUE
  )

  # Artistas que alguna vez tuvieron cada variable.
  available_artists <- d[
    !is.na(value),
    .(
      artists_ever_with_value = uniqueN(artist_id)
    ),
    by = variable
  ]

  # Presencia mensual.
  monthly_present <- d[
    !is.na(value),
    .(
      artists_with_value = uniqueN(artist_id),

      artists_with_original_value = uniqueN(
        artist_id[
          !(interpolated %in% TRUE) &
            !(extrapolated %in% TRUE)
        ]
      ),

      interpolated_values = sum(
        interpolated %in% TRUE
      ),

      extrapolated_values = sum(
        extrapolated %in% TRUE
      )
    ),
    by = .(
      month,
      variable
    )
  ]

  coverage <- CJ(
    month = months,
    variable = variables,
    unique = TRUE
  )

  coverage <- merge(
    coverage,
    available_artists,
    by = "variable",
    all.x = TRUE
  )

  coverage <- merge(
    coverage,
    monthly_present,
    by = c(
      "month",
      "variable"
    ),
    all.x = TRUE
  )

  coverage[
    is.na(artists_ever_with_value),
    artists_ever_with_value := 0L
  ]

  coverage[
    is.na(artists_with_value),
    artists_with_value := 0L
  ]

  coverage[
    is.na(artists_with_original_value),
    artists_with_original_value := 0L
  ]

  coverage[
    is.na(interpolated_values),
    interpolated_values := 0L
  ]

  coverage[
    is.na(extrapolated_values),
    extrapolated_values := 0L
  ]

  coverage[
    ,
    n_artists_total := n_artists
  ]

  coverage[
    ,
    coverage_all_artists_pct :=
      100 *
      artists_with_value /
      n_artists_total
  ]

  coverage[
    ,
    original_coverage_all_artists_pct :=
      100 *
      artists_with_original_value /
      n_artists_total
  ]

  coverage[
    ,
    coverage_available_artists_pct :=
      fifelse(
        artists_ever_with_value > 0,
        100 *
          artists_with_value /
          artists_ever_with_value,
        NA_real_
      )
  ]

  coverage[
    ,
    pct_interpolated_among_values :=
      fifelse(
        artists_with_value > 0,
        100 *
          interpolated_values /
          artists_with_value,
        NA_real_
      )
  ]

  coverage[
    ,
    pct_extrapolated_among_values :=
      fifelse(
        artists_with_value > 0,
        100 *
          extrapolated_values /
          artists_with_value,
        NA_real_
      )
  ]

  coverage[
    ,
    source_family := fifelse(
      grepl(
        "^fan_stats__",
        variable
      ),
      "Fan Stats",
      fifelse(
        grepl(
          "^playlist_evolution__",
          variable
        ),
        "Playlist Evolution",
        "Otra"
      )
    )
  ]

  setorder(
    coverage,
    source_family,
    variable,
    month
  )

  coverage
}

integrated_monthly_coverage <- build_integrated_monthly_coverage(
  monthly = pilot2_monthly,
  ids = PILOT2_ARTIST_IDS,
  variables = REQUIRED_VARIABLES,
  since = FAN_SINCE,
  until = END_DATE
)

In [ ]:
# Cobertura mediana y máxima de cada variable en todo el período

integrated_monthly_coverage[
  ,
  .(
    primer_mes_con_datos =
      if (any(coverage_all_artists_pct > 0)) {
        min(month[coverage_all_artists_pct > 0])
      } else {
        as.Date(NA)
      },
    cobertura_mediana = median(coverage_all_artists_pct),
    cobertura_maxima = max(coverage_all_artists_pct)
  ),
  by = variable
][order(variable)]

In [ ]:
# ============================================================
# RESUMEN DE COBERTURA POR VARIABLE
# PERÍODO COMPLETO
# ============================================================

integrated_variable_summary <-
  integrated_monthly_coverage[
    ,
    .(
      artistas_totales =
        max(n_artists_total),

      artistas_con_algún_dato =
        max(artists_ever_with_value),

      pct_artistas_con_algún_dato =
        100 *
        max(artists_ever_with_value) /
        max(n_artists_total),

      primer_mes_con_datos =
        if (
          any(
            artists_with_value > 0
          )
        ) {

          min(
            month[
              artists_with_value > 0
            ]
          )

        } else {

          as.Date(NA)
        },

      ultimo_mes_con_datos =
        if (
          any(
            artists_with_value > 0
          )
        ) {

          max(
            month[
              artists_with_value > 0
            ]
          )

        } else {

          as.Date(NA)
        },

      cobertura_absoluta_media =
        mean(
          coverage_all_artists_pct
        ),

      cobertura_absoluta_mediana =
        median(
          coverage_all_artists_pct
        ),

      cobertura_absoluta_p10 =
        as.numeric(
          quantile(
            coverage_all_artists_pct,
            probs = 0.10,
            na.rm = TRUE
          )
        ),

      cobertura_absoluta_minima =
        min(
          coverage_all_artists_pct
        ),

      cobertura_condicional_mediana =
        if (
          all(
            is.na(
              coverage_available_artists_pct
            )
          )
        ) {

          NA_real_

        } else {

          median(
            coverage_available_artists_pct,
            na.rm = TRUE
          )
        },

      cobertura_condicional_p10 =
        if (
          all(
            is.na(
              coverage_available_artists_pct
            )
          )
        ) {

          NA_real_

        } else {

          as.numeric(
            quantile(
              coverage_available_artists_pct,
              probs = 0.10,
              na.rm = TRUE
            )
          )
        },

      pct_meses_cobertura_absoluta_80 =
        100 *
        mean(
          coverage_all_artists_pct >= 80
        ),

      pct_interpolado =
        fifelse(
          sum(artists_with_value) > 0,
          100 *
            sum(interpolated_values) /
            sum(artists_with_value),
          NA_real_
        ),

      pct_extrapolado =
        fifelse(
          sum(artists_with_value) > 0,
          100 *
            sum(extrapolated_values) /
            sum(artists_with_value),
          NA_real_
        )
    ),
    by = .(
      source_family,
      variable
    )
  ][
    order(
      source_family,
      -cobertura_absoluta_media
    )
  ]


integrated_variable_summary

## 3. Cobertura conjunta de Fan Stats

Porcentaje de artistas con **todas** las métricas de Fan Stats en cada mes. Exigir todas las plataformas a la vez deja afuera a la mayoría de los artistas: de ahí el recorte de variables de la sección 5.

In [ ]:
# ============================================================
# COBERTURA CONJUNTA
#
# La
# grilla de referencia es artista x variable x mes COMPLETA
# (todos los ids, todas las `variables` pedidas), no sólo los
# pares artista-variable que alguna vez tuvieron un dato.
#
# Si se arma la grilla sólo con pares ya observados, un artista
# que nunca tuvo, por ejemplo, Twitter, deja de contarse esa
# variable como "esperada" — y puede aparecer con
# all_variables_present = TRUE sin haber tenido Twitter nunca.
# Eso infla la cobertura conjunta. Por eso la grilla acá es
# siempre completa.
# ============================================================

build_joint_completeness <- function(
  monthly,
  ids,
  variables,
  since,
  until
) {

  ids <- unique(
    as.character(ids)
  )

  variables <- unique(
    as.character(variables)
  )

  if (
    length(ids) == 0 ||
      length(variables) == 0
  ) {
    return(data.table())
  }

  first_month <- as.Date(
    format(
      since,
      "%Y-%m-01"
    )
  )

  last_month <- as.Date(
    format(
      until,
      "%Y-%m-01"
    )
  )

  months <- seq(
    first_month,
    last_month,
    by = "month"
  )

  full_grid <- CJ(
    artist_id = ids,
    variable = variables,
    month = months
  )

  present <- unique(
    monthly[
      artist_id %in% ids &
        variable %in% variables &
        month %in% months &
        !is.na(value),
      .(
        artist_id,
        variable,
        month
      )
    ]
  )

  present[
    ,
    present := 1L
  ]

  full <- merge(
    full_grid,
    present,
    by = c(
      "artist_id",
      "variable",
      "month"
    ),
    all.x = TRUE
  )

  full[
    is.na(present),
    present := 0L
  ]

  artist_month <- full[
    ,
    .(
      n_variables_expected = .N,
      n_variables_present = sum(present),
      all_variables_present = sum(present) == .N
    ),
    by = .(
      artist_id,
      month
    )
  ]

  artist_month[
    ,
    .(
      artists_total = length(ids),
      artists_complete = sum(all_variables_present),
      joint_coverage_pct =
        100 *
        sum(all_variables_present) /
        length(ids)
    ),
    by = month
  ][
    order(month)
  ]
}

joint_coverage_fan <- build_joint_completeness(
  monthly = pilot2_monthly,
  ids = PILOT2_ARTIST_IDS,
  variables = FAN_REQUIRED_VARIABLES,
  since = FAN_SINCE,
  until = END_DATE
)

joint_coverage_fan

In [ ]:
# Resumen de la cobertura conjunta a lo largo de todo el período
joint_coverage_fan[
  ,
  .(
    primer_mes = min(month),
    ultimo_mes = max(month),
    cobertura_minima = min(joint_coverage_pct),
    cobertura_mediana = median(joint_coverage_pct),
    cobertura_maxima = max(joint_coverage_pct)
  )
]

In [ ]:
# Evolución de la cobertura conjunta de Fan Stats
grafico_cobertura_conjunta_fan <- ggplot(joint_coverage_fan, aes(x = month, y = joint_coverage_pct)) +
  geom_area(fill = "#F18046", alpha = 0.35) +
  geom_line(colour = "#d34101", linewidth = 1) +
  scale_y_continuous(limits = c(0, 100), labels = function(x) paste0(x, "%")) +
  scale_x_date(date_breaks = "6 months", date_labels = "%Y-%m") +
  labs(
    title = "Cobertura conjunta de Fan Stats",
    subtitle = "Artistas con las nueve métricas de Fan Stats presentes en el mes",
    x = NULL,
    y = "Artistas (%)"
  ) +
  theme_minimal(base_size = 13) +
  theme(panel.grid.minor = element_blank(), axis.text.x = element_text(angle = 45, hjust = 1))

print(grafico_cobertura_conjunta_fan)

ggsave(
  file.path(FIGURES_DIR, "cobertura_conjunta_fan_stats.png"),
  grafico_cobertura_conjunta_fan,
  width = 12, height = 6, dpi = 300, bg = "white"
)

### Diagnóstico del pozo de noviembre 2023 a febrero 2024

La cobertura conjunta cae casi a cero durante cuatro meses. El pozo no es general del dataset: lo causa Twitter, cuya cobertura se desploma mientras las demás plataformas se mantienen.

In [ ]:
# ============================================================
# COBERTURA POR VARIABLE ALREDEDOR DEL POZO
# Una fila por variable, una columna por mes, y la caída entre
# octubre y noviembre de 2023 en puntos porcentuales.
# ============================================================

MESES_POZO <- seq(as.Date("2023-09-01"), as.Date("2024-03-01"), by = "month")

cobertura_alrededor_pozo <- dcast(
  integrated_monthly_coverage[
    month %in% MESES_POZO,
    .(variable, mes = format(month, "%Y-%m"), cobertura = round(coverage_all_artists_pct, 1))
  ],
  variable ~ mes,
  value.var = "cobertura"
)

cobertura_alrededor_pozo[, caida_oct_nov_2023 := `2023-11` - `2023-10`]

cobertura_alrededor_pozo[order(caida_oct_nov_2023)]

In [ ]:
integrated_monthly_coverage[
  variable == "fan_stats__twitter__followers" &
    month >= as.Date("2022-01-01"),
  .(
    month,
    coverage_pct = coverage_all_artists_pct
  )
][order(month)]

## 4. Exploración: cobertura individual y conjunta de Fan Stats

Sin Twitter ni TikTok, la cobertura conjunta de Spotify, YouTube e Instagram supera el umbral de referencia desde 2022.

In [ ]:
# ============================================================
# COBERTURA MENSUAL: 5 VARIABLES INDIVIDUALES
# + COBERTURA CONJUNTA DE SPOTIFY + YOUTUBE + INSTAGRAM
# ============================================================

# ============================================================
# 1. VARIABLES
# ============================================================

CUSTOM_VARIABLE_SET <- c(
  "fan_stats__spotify__followers",
  "fan_stats__youtube_channel__subscribers",
  "fan_stats__instagram__followers",
  "fan_stats__tiktok__followers",
  "fan_stats__twitter__followers"
)


# Variables utilizadas para la cobertura conjunta.
# TikTok y Twitter se muestran individualmente,
# pero NO participan de la cobertura conjunta.

VARIABLES_CONJUNTAS <- c(
  "fan_stats__spotify__followers",
  "fan_stats__youtube_channel__subscribers",
  "fan_stats__instagram__followers"
)


# ============================================================
# 2. CONTROLES
# ============================================================

stopifnot(
  "Hay variables fuera de REQUIRED_VARIABLES" =
    all(CUSTOM_VARIABLE_SET %in% REQUIRED_VARIABLES),

  "Se esperaban cinco variables individuales" =
    length(CUSTOM_VARIABLE_SET) == 5L,

  "Se esperaban tres variables para la cobertura conjunta" =
    length(VARIABLES_CONJUNTAS) == 3L
)


# ============================================================
# 3. COBERTURA DE CADA VARIABLE POR SEPARADO
# ============================================================

custom_individual_coverage <- rbindlist(
  lapply(
    CUSTOM_VARIABLE_SET,
    function(v) {

      resultado <- build_joint_completeness(
        monthly = pilot2_monthly,
        ids = PILOT2_ARTIST_IDS,
        variables = v,
        since = FAN_SINCE,
        until = END_DATE
      )

      resultado[
        ,
        .(
          month,
          variable = v,
          coverage_pct = joint_coverage_pct
        )
      ]
    }
  )
)


# ============================================================
# 4. COBERTURA CONJUNTA
#    SPOTIFY + YOUTUBE + INSTAGRAM
# ============================================================

custom_joint_coverage <- build_joint_completeness(
  monthly = pilot2_monthly,
  ids = PILOT2_ARTIST_IDS,
  variables = VARIABLES_CONJUNTAS,
  since = FAN_SINCE,
  until = END_DATE
)


# ============================================================
# 5. NOMBRES BREVES PARA EL GRÁFICO
# ============================================================

SERIES_LABELS <- c(
  "fan_stats__spotify__followers" =
    "Spotify followers",

  "fan_stats__youtube_channel__subscribers" =
    "YouTube subscribers",

  "fan_stats__instagram__followers" =
    "Instagram followers",

  "fan_stats__tiktok__followers" =
    "TikTok followers",

  "fan_stats__twitter__followers" =
    "Twitter followers"
)


# ============================================================
# 6. DATOS PARA EL GRÁFICO
# ============================================================

individual_plot <- custom_individual_coverage[
  ,
  .(
    month,
    serie = unname(SERIES_LABELS[variable]),
    coverage_pct
  )
]


joint_plot <- custom_joint_coverage[
  ,
  .(
    month,
    serie = "Conjunta: Spotify + YouTube + Instagram",
    coverage_pct = joint_coverage_pct
  )
]


coverage_plot_data <- rbindlist(
  list(
    individual_plot,
    joint_plot
  )
)


# ============================================================
# 7. COLORES
# ============================================================

SERIES_COLORS <- c(
  "Spotify followers" = "#0072B2",
  "YouTube subscribers" = "#f6d70c",
  "Instagram followers" = "#CC79A7",
  "TikTok followers" = "#009E73",
  "Twitter followers" = "#870f1b",
  "Conjunta: Spotify + YouTube + Instagram" = "black"
)


# ============================================================
# 8. TIPOS DE LÍNEA
# ============================================================

SERIES_LINE_TYPES <- c(
  "Spotify followers" = "solid",
  "YouTube subscribers" = "solid",
  "Instagram followers" = "solid",
  "TikTok followers" = "solid",
  "Twitter followers" = "solid",
  "Conjunta: Spotify + YouTube + Instagram" = "dashed"
)


# ============================================================
# 9. RANGO TEMPORAL
# ============================================================

first_month <- as.Date(
  format(FAN_SINCE, "%Y-%m-01")
)

last_month <- as.Date(
  format(END_DATE, "%Y-%m-01")
)


# ============================================================
# 10. TAMAÑO DEL GRÁFICO EN NOTEBOOK
# ============================================================

options(
  repr.plot.width = 16,
  repr.plot.height = 8
)


# ============================================================
# 11. GRÁFICO
# ============================================================

coverage_plot <- ggplot(
  coverage_plot_data,
  aes(
    x = month,
    y = coverage_pct,
    color = serie,
    linetype = serie
  )
) +

  geom_line(
    linewidth = 1.1,
    na.rm = TRUE
  ) +

  geom_hline(
    yintercept = COVERAGE_THRESHOLD,
    color = "grey55",
    linetype = "dotted",
    linewidth = 0.7
  ) +

  scale_color_manual(
    name = "Serie",
    values = SERIES_COLORS,
    breaks = names(SERIES_COLORS)
  ) +

  scale_linetype_manual(
    name = "Serie",
    values = SERIES_LINE_TYPES,
    breaks = names(SERIES_LINE_TYPES)
  ) +

  scale_x_date(
    breaks = seq(
      first_month,
      last_month,
      by = "3 months"
    ),
    date_labels = "%Y-%m"
  ) +

  scale_y_continuous(
    breaks = seq(0, 100, by = 5),
    labels = function(x) paste0(x, "%")
  ) +

  coord_cartesian(
    xlim = c(first_month, last_month),
    ylim = c(0, 100)
  ) +

  labs(
    title = "Cobertura mensual de variables y cobertura conjunta",

    subtitle = paste0(
      "Cobertura conjunta: Spotify + YouTube + Instagram · ",
      "Línea horizontal: umbral del ",
      COVERAGE_THRESHOLD,
      "%"
    ),

    x = "Mes",
    y = "Artistas con datos (%)"
  ) +

  theme_minimal(
    base_size = 13
  ) +

  theme(
    axis.text.x = element_text(
      angle = 60,
      hjust = 1,
      size = 9
    ),

    panel.grid.minor = element_blank(),

    plot.background = element_rect(fill = "white", colour = NA),

    legend.position = "bottom",

    legend.title = element_blank()
  ) +

  guides(
    color = guide_legend(
      nrow = 2,
      byrow = TRUE
    ),

    linetype = guide_legend(
      nrow = 2,
      byrow = TRUE
    )
  )


print(coverage_plot)


# ============================================================
# 12. PORCENTAJES EXACTOS POR MES
# ============================================================

coverage_plot_data[
  order(serie, month)
]


# ============================================================
# 13. GUARDAR GRÁFICO
# ============================================================

ggsave(
  filename = file.path(
    getwd(),
    "figuras",
    "cobertura_mensual_variables.png"
  ),

  plot = coverage_plot,

  width = 16,
  height = 8,
  units = "in",
  dpi = 300,
  bg = "white"
)

### Cobertura de TikTok frente al crecimiento de la plataforma

La cobertura de TikTok crece al ritmo de la adopción de la plataforma, no por una mejora de la fuente: por eso queda excluida. Usuarios globales: [Business of Apps](https://www.businessofapps.com/data/tik-tok-statistics/).

In [ ]:
# ============================================================
# USUARIOS GLOBALES VS. ARTISTAS CON DATOS DE TIKTOK
# Q2 de 2021 a 2025. Todos los artistas de la muestra.
# ============================================================

ids <- unique(as.character(PILOT2_ARTIST_IDS))
anios <- 2021:2025
meses_junio <- as.Date(sprintf("%d-06-01", anios))

usuarios_globales <- data.table(
  year = anios,
  usuarios_millones = c(902, 1366, 1587, 1685, 1844)
)

# Contar, en cada junio, cuántos artistas de TODA la muestra
# tienen un valor registrado para TikTok followers.
# Un valor 0 cuenta como dato; NA no cuenta.

artistas_por_anio <- as.data.table(pilot2_monthly)[
  as.character(artist_id) %in% ids &
    variable == "fan_stats__tiktok__followers" &
    month %in% meses_junio &
    !is.na(value),
  .(
    artistas_con_dato = uniqueN(artist_id)
  ),
  by = .(
    year = as.integer(format(month, "%Y"))
  )
]

comparacion_tiktok <- merge(
  usuarios_globales,
  artistas_por_anio,
  by = "year",
  all.x = TRUE
)

comparacion_tiktok[
  is.na(artistas_con_dato),
  artistas_con_dato := 0L
]

comparacion_tiktok[
  ,
  `:=`(
    artistas_totales = length(ids),
    cobertura_pct = 100 * artistas_con_dato / length(ids)
  )
]

if (comparacion_tiktok[year == 2021, artistas_con_dato] == 0L) {
  stop("En junio de 2021 no hay artistas con dato: no puede calcularse el índice.")
}

# Poner ambas series en 2021 = 100 para comparar
# cuánto crece cada una, pese a sus distintas magnitudes.

comparacion_tiktok[
  ,
  `:=`(
    indice_usuarios =
      100 * usuarios_millones / usuarios_millones[1],

    indice_artistas_con_dato =
      100 * artistas_con_dato / artistas_con_dato[1]
  )
]

series_grafico <- rbindlist(
  list(
    comparacion_tiktok[
      ,
      .(
        year,
        serie = "Usuarios globales de TikTok",
        indice = indice_usuarios
      )
    ],
    comparacion_tiktok[
      ,
      .(
        year,
        serie = "Artistas con dato de TikTok",
        indice = indice_artistas_con_dato
      )
    ]
  )
)

grafico_tendencias_tiktok <- ggplot(
  series_grafico,
  aes(x = year, y = indice, color = serie)
) +
  geom_line(linewidth = 1.2) +
  geom_point(size = 3) +
  scale_x_continuous(breaks = anios) +
  scale_color_manual(
    values = c(
      "Usuarios globales de TikTok" = "#0072B2",
      "Artistas con dato de TikTok" = "#009E73"
    )
  ) +
  labs(
    title = "Crecimiento de usuarios y de artistas con datos de TikTok",
    subtitle = paste0(
      "Índice 2021 = 100 · muestra de ",
      length(ids),
      " artistas"
    ),
    x = "Año",
    y = "Índice de crecimiento",
    color = NULL,
    caption = paste(
      "Usuarios globales: Business of Apps, Q2.",
      "Artistas con dato: Chartmetric, junio de cada año."
    )
  ) +
  theme_minimal(base_size = 13) +
  theme(
    legend.position = "bottom",
    plot.background = element_rect(fill = "white", colour = NA)
  )

print(grafico_tendencias_tiktok)

ggsave(
  file.path(FIGURES_DIR, "tiktok_usuarios_vs_artistas_con_dato.png"),
  grafico_tendencias_tiktok,
  width = 10, height = 6, dpi = 300, bg = "white"
)

comparacion_tiktok[
  ,
  .(
    year,
    usuarios_millones,
    artistas_con_dato,
    artistas_totales,
    cobertura_pct = round(cobertura_pct, 1),
    indice_usuarios = round(indice_usuarios, 1),
    indice_artistas_con_dato = round(indice_artistas_con_dato, 1)
  )
]

## 5. Panel 2022-2026

Decisiones del recorte:

- **Marco:** los 5000 artistas mejor rankeados del dataset maestro, con deciles propios (`decil_marco`).
- **Ventana:** marzo 2022 a marzo 2026, 49 meses (cuatro ciclos anuales completos).
- **Variables (5):** `spotify_followers`, `instagram_followers`, `spotify_playlist_count`, `youtube_channel_subscribers` y `youtube_playlist_count`. De cada par equivalente se conserva la de mejor cobertura; Amazon queda afuera porque baja la retención en los deciles bajos.
- **Artistas (criterio de panel, cobertura conjunta por artista):** dato en al menos 40 de los 49 meses (80%) en cada una de las 5 variables a la vez (`cobertura_minima` >= `UMBRAL_PROPORCION_MESES_ARTISTA`).
- **NA:** no se imputa. Cada celda se clasifica en `tipo_dato`: `observado`, `hueco_interno` (interpolable), `previo_primer_dato` o `posterior_ultimo_dato`.

In [ ]:
# ============================================================
# DATASET CONSOLIDADO 2022-2026: VENTANA, MARCO, VARIABLES Y
# UMBRAL
#
# Ventana: marzo 2022 a marzo 2026, ambos inclusive (49 meses).
# Marco de muestreo: los MARCO_TOP_N artistas mejor rankeados
# (cm_artist_rank) del dataset maestro, con deciles propios.
# Variables: 5, elegidas por cobertura total y por decil (ver
# markdown de la sección). En los pares con dos candidatas se
# conserva la de mejor cobertura.
# La tabla de mapeo documenta el nombre original (con el
# prefijo del endpoint de Chartmetric) y el nombre nuevo.
# ============================================================

VENTANA_INICIO_2022_03 <- as.Date("2022-03-01")
VENTANA_FIN_2026_03 <- as.Date("2026-03-01")

MESES_VENTANA_22_26 <- seq(
  VENTANA_INICIO_2022_03,
  VENTANA_FIN_2026_03,
  by = "month"
)

# Proporción mínima de meses con dato que cada artista debe
# tener en cada una de las variables para entrar al panel
UMBRAL_PROPORCION_MESES_ARTISTA <- 0.8

mapeo_nombres_variables_22_26 <- data.table(
  variable_original = c(
    "fan_stats__spotify__followers",
    "fan_stats__instagram__followers",
    "playlist_evolution__spotify__playlist_count",
    "fan_stats__youtube_channel__subscribers",
    "playlist_evolution__youtube__playlist_count"
  ),
  variable_nombre = c(
    "spotify_followers",
    "instagram_followers",
    "spotify_playlist_count",
    "youtube_channel_subscribers",
    "youtube_playlist_count"
  )
)

# Pares de candidatas: la elegida (en el mapeo) y la descartada.
# La celda de cobertura controla que la elegida siga siendo la
# de mejor cobertura a medida que crece la muestra.
pares_variables_22_26 <- data.table(
  grupo = c("spotify_playlist", "youtube_channel", "youtube_playlist"),
  variable_elegida = c(
    "playlist_evolution__spotify__playlist_count",
    "fan_stats__youtube_channel__subscribers",
    "playlist_evolution__youtube__playlist_count"
  ),
  variable_descartada = c(
    "playlist_evolution__spotify__playlist_total_reach",
    "fan_stats__youtube_channel__views",
    "playlist_evolution__youtube__playlist_total_reach"
  )
)

# Variable evaluada y excluida por bajar la cobertura
VARIABLES_EVALUADAS_EXCLUIDAS_22_26 <- c(
  "playlist_evolution__amazon__playlist_count"
)

stopifnot(
  "Hay variables del mapeo ausentes en pilot2_monthly" =
    all(
      mapeo_nombres_variables_22_26$variable_original %in%
        unique(pilot2_monthly$variable)
    ),
  "Hay nombres nuevos repetidos" =
    !anyDuplicated(mapeo_nombres_variables_22_26$variable_nombre),
  "Las variables elegidas de los pares no están en el mapeo" =
    all(
      pares_variables_22_26$variable_elegida %in%
        mapeo_nombres_variables_22_26$variable_original
    )
)

N_MESES_VENTANA_22_26 <- length(MESES_VENTANA_22_26)
N_VARIABLES_PANEL_22_26 <- nrow(mapeo_nombres_variables_22_26)
MINIMO_MESES_CON_DATO_22_26 <- UMBRAL_PROPORCION_MESES_ARTISTA * N_MESES_VENTANA_22_26

cat(
  "Meses en la ventana:", N_MESES_VENTANA_22_26, "\n",
  "Marco de muestreo: top", MARCO_TOP_N, "del dataset maestro\n",
  "Variables:", N_VARIABLES_PANEL_22_26, "\n",
  "Mínimo de meses con dato por artista y variable:",
  ceiling(MINIMO_MESES_CON_DATO_22_26), "\n\n"
)

mapeo_nombres_variables_22_26

In [ ]:
# ============================================================
# MARCO DE MUESTREO: TOP MARCO_TOP_N Y DECILES DEL MARCO
#
# El marco pasa a ser los MARCO_TOP_N artistas mejor rankeados
# del dataset maestro, con deciles recalculados en esa escala
# (decil_marco). El decil sobre los 7000 (decile de rank_data)
# se conserva como referencia (decil_7000).
# La muestra descargada se sorteó con la misma densidad dentro
# de cada decil de los 7000, así que los artistas descargados
# que caen dentro del marco siguen siendo una muestra aleatoria
# uniforme del marco. No hace falta volver a sortear. Las
# ampliaciones posteriores se sortean sólo dentro del marco
# (celda AMPLIACIÓN DENTRO DEL MARCO TOP 5000).
# ============================================================

# Marco construido en el notebook 01 (insumos), el mismo que
# usó la ampliación de la muestra
marco_top_22_26 <- copy(MARCO_TOP)

ARTISTAS_DESCARGADOS_22_26_IDS <- unique(as.character(PILOT2_ARTIST_IDS))

ARTISTAS_MUESTRA_MARCO_22_26_IDS <- intersect(
  ARTISTAS_DESCARGADOS_22_26_IDS,
  marco_top_22_26$ID
)

N_ARTISTAS_DESCARGADOS_22_26 <- length(ARTISTAS_DESCARGADOS_22_26_IDS)
N_ARTISTAS_MUESTRA_MARCO_22_26 <- length(ARTISTAS_MUESTRA_MARCO_22_26_IDS)

cat(
  "Artistas descargados:", N_ARTISTAS_DESCARGADOS_22_26, "\n",
  "Artistas descargados dentro del marco top", MARCO_TOP_N, ":",
  N_ARTISTAS_MUESTRA_MARCO_22_26, "\n",
  "Rank máximo del marco:", max(marco_top_22_26$cm_artist_rank), "\n\n"
)

# Deciles del marco: tamaño en el marco y muestra descargada
marco_top_22_26[
  ,
  .(
    artistas_marco = .N,
    muestra_descargada = sum(ID %in% ARTISTAS_MUESTRA_MARCO_22_26_IDS),
    rank_min = min(cm_artist_rank),
    rank_max = max(cm_artist_rank)
  ),
  by = decil_marco
][order(decil_marco)]

In [ ]:
# ============================================================
# COBERTURA POR VARIABLE EN LA VENTANA, TOTAL Y POR DECIL
#
# Base de la elección de variables, sobre los artistas
# descargados dentro del marco:
# - artistas_sobre_umbral: proporción de artistas con
#   UMBRAL_PROPORCION_MESES_ARTISTA o más de los meses cubiertos,
#   en total y por decil del marco;
# - cobertura_celdas: proporción de celdas artista-mes con dato.
# Se muestran las variables del panel, las descartadas de cada
# par y las evaluadas y excluidas.
# Control: en cada par, la variable elegida tiene que seguir
# siendo la de mejor cobertura (total y en el decil más bajo).
# ============================================================

# Meses con dato por artista y variable, para todos los artistas
# descargados y todas las variables (se reutiliza en la celda de
# selección)
meses_con_dato_por_artista_variable_22_26 <- unique(
  pilot2_monthly[
    artist_id %in% ARTISTAS_DESCARGADOS_22_26_IDS &
      month %in% MESES_VENTANA_22_26 &
      !is.na(value),
    .(artist_id, variable, month)
  ]
)[
  ,
  .(meses_con_dato = .N),
  by = .(artist_id, variable)
]

variables_evaluadas_22_26 <- c(
  mapeo_nombres_variables_22_26$variable_original,
  pares_variables_22_26$variable_descartada,
  VARIABLES_EVALUADAS_EXCLUIDAS_22_26
)

# Grilla artista del marco x variable evaluada (0 meses si no hay dato)
cobertura_marco_variable_22_26 <- CJ(
  artist_id = ARTISTAS_MUESTRA_MARCO_22_26_IDS,
  variable = variables_evaluadas_22_26
)

cobertura_marco_variable_22_26 <- merge(
  cobertura_marco_variable_22_26,
  meses_con_dato_por_artista_variable_22_26,
  by = c("artist_id", "variable"),
  all.x = TRUE
)

cobertura_marco_variable_22_26[is.na(meses_con_dato), meses_con_dato := 0L]

cobertura_marco_variable_22_26 <- merge(
  cobertura_marco_variable_22_26,
  marco_top_22_26[, .(ID, decil_marco)],
  by.x = "artist_id",
  by.y = "ID"
)

cobertura_marco_variable_22_26[
  ,
  sobre_umbral := meses_con_dato >= MINIMO_MESES_CON_DATO_22_26
]

cobertura_por_variable_22_26 <- cobertura_marco_variable_22_26[
  ,
  .(
    cobertura_celdas = round(mean(meses_con_dato) / N_MESES_VENTANA_22_26, 3),
    artistas_sobre_umbral = round(mean(sobre_umbral), 3)
  ),
  by = variable
]

cobertura_por_variable_decil_22_26 <- dcast(
  cobertura_marco_variable_22_26[
    ,
    .(p = round(mean(sobre_umbral), 3)),
    by = .(variable, decil_marco)
  ],
  variable ~ paste0("d", decil_marco),
  value.var = "p"
)

cobertura_por_variable_22_26 <- merge(
  cobertura_por_variable_22_26,
  cobertura_por_variable_decil_22_26,
  by = "variable"
)

cobertura_por_variable_22_26[
  ,
  rol := fcase(
    variable %in% mapeo_nombres_variables_22_26$variable_original, "panel",
    variable %in% pares_variables_22_26$variable_descartada, "descartada_por_par",
    default = "evaluada_excluida"
  )
]

setcolorder(
  cobertura_por_variable_22_26,
  c("variable", "rol", "cobertura_celdas", "artistas_sobre_umbral", paste0("d", 1:10))
)

cat(
  "Artistas descargados dentro del marco:", N_ARTISTAS_MUESTRA_MARCO_22_26, "\n",
  "Proporción de artistas con", UMBRAL_PROPORCION_MESES_ARTISTA,
  "o más de los meses cubiertos, total y por decil del marco:\n\n"
)

print(cobertura_por_variable_22_26[order(rol, -artistas_sobre_umbral)])

# Control de pares: la elegida tiene que cubrir al menos lo mismo
# que la descartada, en total y en el decil 10
control_pares_22_26 <- pares_variables_22_26[
  ,
  .(
    grupo,
    variable_elegida,
    variable_descartada,
    elegida_total = cobertura_por_variable_22_26[match(variable_elegida, variable), artistas_sobre_umbral],
    descartada_total = cobertura_por_variable_22_26[match(variable_descartada, variable), artistas_sobre_umbral],
    elegida_d10 = cobertura_por_variable_22_26[match(variable_elegida, variable), d10],
    descartada_d10 = cobertura_por_variable_22_26[match(variable_descartada, variable), d10]
  )
]

cat("\nControl de pares:\n")
print(control_pares_22_26[, .(grupo, elegida_total, descartada_total, elegida_d10, descartada_d10)])

pares_invertidos_22_26 <- control_pares_22_26[
  descartada_total > elegida_total | descartada_d10 > elegida_d10,
  grupo
]

if (length(pares_invertidos_22_26) > 0) {
  warning(
    "En estos pares la variable descartada ahora cubre más que la elegida: ",
    paste(pares_invertidos_22_26, collapse = ", "),
    ". Revisar mapeo_nombres_variables_22_26 y pares_variables_22_26."
  )
} else {
  cat("\nEn todos los pares la variable elegida sigue siendo la de mejor cobertura.\n")
}

In [ ]:
# ============================================================
# COBERTURA POR ARTISTA Y SELECCIÓN DEL PANEL
#
# cobertura_por_artista_22_26: una fila por artista descargado
# (dentro y fuera del marco), con la proporción de meses
# cubiertos en cada variable del panel, la mínima entre ellas,
# el decil sobre los 7000, si está en el marco, su decil del
# marco y si entra al panel. Se exporta junto al panel: registra
# el recorte y permite análisis de sensibilidad con los
# artistas de fuera del marco.
# Entran al panel los artistas del marco cuya cobertura mínima
# alcanza UMBRAL_PROPORCION_MESES_ARTISTA.
# Reutiliza meses_con_dato_por_artista_variable_22_26 de la
# celda de cobertura por variable.
# ============================================================

cobertura_por_artista_22_26 <- CJ(
  artist_id = ARTISTAS_DESCARGADOS_22_26_IDS,
  variable = mapeo_nombres_variables_22_26$variable_original
)

cobertura_por_artista_22_26 <- merge(
  cobertura_por_artista_22_26,
  meses_con_dato_por_artista_variable_22_26,
  by = c("artist_id", "variable"),
  all.x = TRUE
)

cobertura_por_artista_22_26[is.na(meses_con_dato), meses_con_dato := 0L]

cobertura_por_artista_22_26[
  ,
  variable := mapeo_nombres_variables_22_26$variable_nombre[
    match(variable, mapeo_nombres_variables_22_26$variable_original)
  ]
]

cobertura_por_artista_22_26 <- dcast(
  cobertura_por_artista_22_26[
    ,
    .(artist_id, variable, cobertura = meses_con_dato / N_MESES_VENTANA_22_26)
  ],
  artist_id ~ variable,
  value.var = "cobertura"
)

cobertura_por_artista_22_26[
  ,
  cobertura_minima := do.call(pmin, .SD),
  .SDcols = mapeo_nombres_variables_22_26$variable_nombre
]

# Rank, decil sobre los 7000, marco y decil del marco
cobertura_por_artista_22_26 <- merge(
  cobertura_por_artista_22_26,
  copy(rank_data)[, .(artist_id = as.character(ID), decil_7000 = decile)],
  by = "artist_id",
  all.x = TRUE
)

cobertura_por_artista_22_26 <- merge(
  cobertura_por_artista_22_26,
  marco_top_22_26[, .(artist_id = ID, decil_marco)],
  by = "artist_id",
  all.x = TRUE
)

cobertura_por_artista_22_26[, en_marco := !is.na(decil_marco)]

cobertura_por_artista_22_26[
  ,
  en_panel := en_marco & cobertura_minima >= UMBRAL_PROPORCION_MESES_ARTISTA
]

if (nrow(ARTISTS) > 0) {
  cobertura_por_artista_22_26 <- merge(
    ARTISTS,
    cobertura_por_artista_22_26,
    by.x = "ID",
    by.y = "artist_id",
    all.y = TRUE
  )
  setnames(cobertura_por_artista_22_26, "ID", "artist_id")
}

setcolorder(
  cobertura_por_artista_22_26,
  c(
    intersect(
      c("artist_id", "artist_name", "cm_artist_rank"),
      names(cobertura_por_artista_22_26)
    ),
    "decil_7000",
    "en_marco",
    "decil_marco",
    "en_panel",
    "cobertura_minima",
    mapeo_nombres_variables_22_26$variable_nombre
  )
)

setorder(cobertura_por_artista_22_26, -en_panel, decil_marco, -cobertura_minima, artist_id)

ARTISTAS_PANEL_22_26_IDS <- sort(
  cobertura_por_artista_22_26[en_panel == TRUE, artist_id]
)

N_ARTISTAS_PANEL_22_26 <- length(ARTISTAS_PANEL_22_26_IDS)

cat(
  "Artistas descargados:", N_ARTISTAS_DESCARGADOS_22_26, "\n",
  "Dentro del marco top", MARCO_TOP_N, ":", N_ARTISTAS_MUESTRA_MARCO_22_26, "\n",
  "Entran al panel:", N_ARTISTAS_PANEL_22_26,
  "(", round(100 * N_ARTISTAS_PANEL_22_26 / N_ARTISTAS_MUESTRA_MARCO_22_26, 1),
  "% de los descargados dentro del marco)\n\n"
)

# Cuántos artistas del marco quedarían con otros umbrales
cat("Artistas del marco en el panel según el umbral de cobertura mínima:\n")
print(
  data.table(
    umbral = c(0.6, 0.7, 0.8, 0.9, 1)
  )[
    ,
    artistas := sapply(
      umbral,
      function(u) cobertura_por_artista_22_26[en_marco == TRUE & cobertura_minima >= u, .N]
    )
  ]
)

# Artistas del marco por debajo del umbral en cada variable
# (un mismo artista puede fallar en varias)
cat("\nArtistas del marco por debajo del umbral en cada variable:\n")

data.table(
  variable = mapeo_nombres_variables_22_26$variable_nombre,
  artistas_bajo_umbral = sapply(
    mapeo_nombres_variables_22_26$variable_nombre,
    function(v) {
      cobertura_por_artista_22_26[
        en_marco == TRUE & get(v) < UMBRAL_PROPORCION_MESES_ARTISTA,
        .N
      ]
    }
  )
)[order(-artistas_bajo_umbral)]

In [ ]:
# ============================================================
# ESTRATIFICACIÓN DEL PANEL POR DECIL DEL MARCO
#
# Cómo quedó la composición del panel respecto de la muestra
# descargada dentro del marco:
# - artistas por decil en la muestra y en el panel;
# - tasa de retención por decil y test chi2 de homogeneidad;
# - mediana de rank en la muestra y en el panel, dentro de cada
#   decil (sesgo intra decil);
# - tamaño de un panel balanceado (10 x el mínimo por decil).
# No submuestrea: la decisión de balancear se toma aparte.
# ============================================================

muestra_marco_22_26 <- cobertura_por_artista_22_26[en_marco == TRUE]

estratificacion_panel_22_26 <- muestra_marco_22_26[
  ,
  .(
    muestra = .N,
    panel = sum(en_panel),
    tasa_retencion = round(mean(en_panel), 3),
    rank_mediana_muestra = as.numeric(median(cm_artist_rank)),
    rank_mediana_panel = as.numeric(median(cm_artist_rank[en_panel]))
  ),
  by = decil_marco
][order(decil_marco)]

test_retencion_22_26 <- chisq.test(
  table(muestra_marco_22_26$decil_marco, muestra_marco_22_26$en_panel)
)

MINIMO_PANEL_POR_DECIL_22_26 <- min(estratificacion_panel_22_26$panel)

cat(
  "Panel:", N_ARTISTAS_PANEL_22_26, "artistas de", N_ARTISTAS_MUESTRA_MARCO_22_26,
  "descargados dentro del marco top", MARCO_TOP_N, "\n",
  "Retención decil 10 / decil 1:",
  round(
    estratificacion_panel_22_26[decil_marco == 10, tasa_retencion] /
      estratificacion_panel_22_26[decil_marco == 1, tasa_retencion],
    2
  ), "\n",
  "Chi2 de homogeneidad de la retención entre deciles: p =",
  format.pval(test_retencion_22_26$p.value, digits = 2), "\n",
  "Mínimo de artistas del panel por decil:", MINIMO_PANEL_POR_DECIL_22_26,
  "-> panel balanceado posible:", 10 * MINIMO_PANEL_POR_DECIL_22_26, "artistas\n\n"
)

estratificacion_panel_22_26

In [ ]:
# ============================================================
# PANEL BALANCEADO: GRILLA ARTISTA x VARIABLE x MES
# Una fila por combinación del período, con NA explícitos donde
# no hay dato. El dataset mensual crudo sólo tiene filas con
# dato: la grilla completa se arma recién acá, con la ventana y
# las variables ya fijadas.
# ============================================================

build_balanced_panel <- function(
  monthly,
  ids,
  variables,
  since,
  until
) {

  ids <- unique(
    as.character(ids)
  )

  variables <- unique(
    as.character(variables)
  )

  first_month <- as.Date(
    format(since, "%Y-%m-01")
  )

  last_month <- as.Date(
    format(until, "%Y-%m-01")
  )

  months <- seq(
    first_month,
    last_month,
    by = "month"
  )

  grid <- CJ(
    artist_id = ids,
    variable = variables,
    month = months
  )

  d <- unique(
    monthly[
      artist_id %in% ids &
        variable %in% variables &
        month %in% months,
      .(
        artist_id,
        variable,
        month,
        value,
        interpolated,
        extrapolated
      )
    ],
    by = c("artist_id", "variable", "month"),
    fromLast = TRUE
  )

  panel <- merge(
    grid,
    d,
    by = c("artist_id", "variable", "month"),
    all.x = TRUE
  )

  if (nrow(ARTISTS) > 0) {

    panel <- merge(
      panel,
      ARTISTS,
      by.x = "artist_id",
      by.y = "ID",
      all.x = TRUE
    )
  }

  setorder(panel, artist_id, variable, month)

  panel
}

In [ ]:
# ============================================================
# PANEL BALANCEADO LONG: GRILLA COMPLETA CON NA EXPLÍCITOS
#
# 1. build_balanced_panel() arma una fila por artista x
#    variable x mes y deja NA donde no hay dato.
# 2. Se clasifica cada fila según su posición respecto del
#    primer y último dato del artista en esa variable. Los
#    hitos se calculan sobre todo el historial descargado
#    (desde FAN_SINCE), no sólo sobre la ventana.
# 3. Se renombran las variables según el mapeo.
# 4. Se agrega el decil del marco de cada artista.
# ============================================================

dt_panel_22_26_na_long <- build_balanced_panel(
  monthly = pilot2_monthly,
  ids = ARTISTAS_PANEL_22_26_IDS,
  variables = mapeo_nombres_variables_22_26$variable_original,
  since = VENTANA_INICIO_2022_03,
  until = VENTANA_FIN_2026_03
)

hitos_observacion_historicos_22_26 <- pilot2_monthly[
  artist_id %in% ARTISTAS_PANEL_22_26_IDS &
    variable %in% mapeo_nombres_variables_22_26$variable_original &
    !is.na(value),
  .(
    primer_mes_observado = min(month),
    ultimo_mes_observado = max(month)
  ),
  by = .(artist_id, variable)
]

dt_panel_22_26_na_long <- merge(
  dt_panel_22_26_na_long,
  hitos_observacion_historicos_22_26,
  by = c("artist_id", "variable"),
  all.x = TRUE
)

dt_panel_22_26_na_long[
  ,
  tipo_dato := fcase(
    !is.na(value), "observado",
    month < primer_mes_observado, "previo_primer_dato",
    month > ultimo_mes_observado, "posterior_ultimo_dato",
    default = "hueco_interno"
  )
]

dt_panel_22_26_na_long[
  ,
  variable := mapeo_nombres_variables_22_26$variable_nombre[
    match(variable, mapeo_nombres_variables_22_26$variable_original)
  ]
]

dt_panel_22_26_na_long[
  ,
  c("primer_mes_observado", "ultimo_mes_observado") := NULL
]

# Decil del marco, para estratificar y submuestrear después
dt_panel_22_26_na_long <- merge(
  dt_panel_22_26_na_long,
  marco_top_22_26[, .(artist_id = ID, decil_marco)],
  by = "artist_id",
  all.x = TRUE
)

columnas_id_artista_22_26 <- intersect(
  c("artist_id", "artist_name", "cm_artist_rank", "decil_marco"),
  names(dt_panel_22_26_na_long)
)

setcolorder(
  dt_panel_22_26_na_long,
  c(
    columnas_id_artista_22_26,
    "variable", "month", "value", "tipo_dato",
    "interpolated", "extrapolated"
  )
)

setorder(dt_panel_22_26_na_long, artist_id, variable, month)

cat(
  "Panel long:",
  nrow(dt_panel_22_26_na_long), "filas x",
  ncol(dt_panel_22_26_na_long), "columnas\n\n"
)

head(dt_panel_22_26_na_long)

In [ ]:
# ============================================================
# SANITY CHECK DEL PANEL LONG
#
# - filas = artistas x variables x meses;
# - sin claves artista-variable-mes duplicadas;
# - todas las series cumplen el umbral de meses con dato;
# - composición de tipo_dato por variable;
# - racha más larga de NA consecutivos por serie, que marca
#   cuánto tiene que estirarse la interpolación.
# ============================================================

n_filas_esperadas_long_22_26 <- N_ARTISTAS_PANEL_22_26 *
  N_VARIABLES_PANEL_22_26 *
  N_MESES_VENTANA_22_26

cat(
  "Artistas:", uniqueN(dt_panel_22_26_na_long$artist_id), "\n",
  "Variables:", uniqueN(dt_panel_22_26_na_long$variable), "\n",
  "Meses:", uniqueN(dt_panel_22_26_na_long$month),
  "| rango:", format(range(dt_panel_22_26_na_long$month)), "\n",
  "Filas:", nrow(dt_panel_22_26_na_long),
  "| esperadas:", n_filas_esperadas_long_22_26, "\n",
  "Proporción de NA:",
  round(mean(is.na(dt_panel_22_26_na_long$value)), 4), "\n\n"
)

stopifnot(
  "La cantidad de filas no coincide con la grilla esperada" =
    nrow(dt_panel_22_26_na_long) == n_filas_esperadas_long_22_26,
  "Hay claves artista-variable-mes duplicadas" =
    !anyDuplicated(
      dt_panel_22_26_na_long,
      by = c("artist_id", "variable", "month")
    ),
  "Hay series por debajo del umbral de meses con dato" =
    dt_panel_22_26_na_long[
      ,
      .(meses_con_dato = sum(!is.na(value))),
      by = .(artist_id, variable)
    ][
      ,
      all(meses_con_dato >= UMBRAL_PROPORCION_MESES_ARTISTA * N_MESES_VENTANA_22_26)
    ]
)

composicion_tipo_dato_por_variable_22_26 <- dcast(
  dt_panel_22_26_na_long[, .N, by = .(variable, tipo_dato)],
  variable ~ tipo_dato,
  value.var = "N",
  fill = 0
)

print(composicion_tipo_dato_por_variable_22_26)

# Racha más larga de NA consecutivos en cada serie artista-variable
racha_na_maxima_por_serie_22_26 <- dt_panel_22_26_na_long[
  ,
  {
    rachas <- rle(is.na(value))
    .(
      racha_na_maxima = if (any(rachas$values)) {
        max(rachas$lengths[rachas$values])
      } else {
        0L
      }
    )
  },
  by = .(artist_id, variable)
]

cat("\nSeries según su racha más larga de NA consecutivos:\n")

racha_na_maxima_por_serie_22_26[
  ,
  .(series = .N),
  by = racha_na_maxima
][order(racha_na_maxima)]

In [ ]:
# ============================================================
# PANEL WIDE Y EXPORTACIÓN
#
# Wide: una fila por artista y mes, una columna por variable,
# en el orden del mapeo. tipo_dato, interpolated y extrapolated
# quedan sólo en el long.
# Nombres, con cantidades calculadas en cada corrida:
# - panel: artistas_ST_<n panel>de<n muestra en el marco>_top<marco>_22_26_na
#   _long.csv y _wide.csv;
# - cobertura por artista (todos los descargados, dentro y fuera
#   del marco): artistas_ST_<n descargados>_22_26.csv.
# Se exportan en CSV al directorio del proyecto (los CSV están
# excluidos del repo por .gitignore).
# ============================================================

formula_wide_22_26 <- as.formula(
  paste(
    paste(c(columnas_id_artista_22_26, "month"), collapse = " + "),
    "~ variable"
  )
)

dt_panel_22_26_na_wide <- dcast(
  dt_panel_22_26_na_long,
  formula_wide_22_26,
  value.var = "value"
)

setcolorder(
  dt_panel_22_26_na_wide,
  c(
    columnas_id_artista_22_26,
    "month",
    mapeo_nombres_variables_22_26$variable_nombre
  )
)

setorder(dt_panel_22_26_na_wide, artist_id, month)

# Sanity check: una fila por artista y mes
stopifnot(
  "El wide no tiene una fila por artista y mes" =
    nrow(dt_panel_22_26_na_wide) ==
      N_ARTISTAS_PANEL_22_26 * N_MESES_VENTANA_22_26
)

prefijo_panel_22_26 <- paste0(
  "artistas_ST_", N_ARTISTAS_PANEL_22_26, "de", N_ARTISTAS_MUESTRA_MARCO_22_26,
  "_top", MARCO_TOP_N, "_22_26_na"
)

archivo_panel_long_22_26 <- paste0(prefijo_panel_22_26, "_long.csv")
archivo_panel_wide_22_26 <- paste0(prefijo_panel_22_26, "_wide.csv")
archivo_cobertura_artista_22_26 <- paste0(
  "artistas_ST_", N_ARTISTAS_DESCARGADOS_22_26, "_22_26.csv"
)

fwrite(dt_panel_22_26_na_long, archivo_panel_long_22_26)
fwrite(dt_panel_22_26_na_wide, archivo_panel_wide_22_26)

# Cobertura redondeada sólo en el archivo exportado
columnas_cobertura_22_26 <- c(
  "cobertura_minima",
  mapeo_nombres_variables_22_26$variable_nombre
)

fwrite(
  copy(cobertura_por_artista_22_26)[
    ,
    (columnas_cobertura_22_26) := lapply(.SD, round, 3),
    .SDcols = columnas_cobertura_22_26
  ],
  archivo_cobertura_artista_22_26
)

cat(
  "Long:", archivo_panel_long_22_26, "-",
  nrow(dt_panel_22_26_na_long), "filas x",
  ncol(dt_panel_22_26_na_long), "columnas\n",
  "Wide:", archivo_panel_wide_22_26, "-",
  nrow(dt_panel_22_26_na_wide), "filas x",
  ncol(dt_panel_22_26_na_wide), "columnas\n",
  "Cobertura por artista:", archivo_cobertura_artista_22_26, "-",
  nrow(cobertura_por_artista_22_26), "filas x",
  ncol(cobertura_por_artista_22_26), "columnas\n\n"
)

head(dt_panel_22_26_na_wide)